In [11]:
# CELL 1 — Extract probabilities + embeddings for ROC/PR/calibration/t-SNE
import numpy as np
import torch
import torch.nn as nn
import math
import json
import os
import pennylane as qml
from sklearn.metrics import accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_QUBITS, ENTANGLING_LAYERS, D_MODEL, D_FF, N_HEADS, N_TOKENS = 4, 2, 64, 128, 4, 225
TRAIN_BATCH_SIZE, EPOCHS, LR, WEIGHT_DECAY, GRAD_CLIP_NORM = 32, 50, 2e-3, 1e-4, 1.0
SEED = 42
QUANTUM_DEVICE_NAME, DIFF_METHOD = "default.qubit", "backprop"

def build_quantum_layer():
    dev = qml.device(QUANTUM_DEVICE_NAME, wires=N_QUBITS)
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=ENTANGLING_LAYERS, n_wires=N_QUBITS)
    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]
    return qml.qnn.TorchLayer(circuit, {"weights": weight_shape})

class QuantumTokenEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, N_QUBITS)
        self.q_layer = build_quantum_layer()
        self.out_proj = nn.Linear(N_QUBITS, D_MODEL)
    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, N_QUBITS)
        q_out = self.q_layer(flat).reshape(b, n, N_QUBITS)
        return self.out_proj(q_out)

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_tokens, d_model):
        super().__init__()
        pe = torch.zeros(n_tokens, d_model)
        pos = torch.arange(0, n_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe

class QuantFormerWithEmbeddings(nn.Module):
    def __init__(self, k_dim, n_classes):
        super().__init__()
        self.q_encoder = QuantumTokenEncoder(k_dim)
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.transformer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True, dropout=0.0)
        self.classifier = nn.Linear(D_MODEL, n_classes)
    def forward(self, tokens, return_embedding=False):
        x = self.q_encoder(tokens)
        x = self.pos_enc(x)
        x = self.transformer(x)
        embedding = x.mean(dim=1)
        logits = self.classifier(embedding)
        if return_embedding:
            return logits, embedding
        return logits

def get_param_groups(model):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        (no_decay if "q_layer" in name else decay).append(param)
    return [{"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0}]

def train_full_config(k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels):
    torch.manual_seed(SEED)
    model = QuantFormerWithEmbeddings(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    best_val_acc, best_state = -1, None
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(val_labels, model(val_tokens.to(DEVICE)).argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

def extract_artifacts(model, test_tokens, test_labels, batch_size=256):
    model.eval()
    all_probs, all_embeddings, all_preds = [], [], []
    n = test_tokens.shape[0]
    with torch.no_grad():
        for i in range(0, n, batch_size):
            xb = test_tokens[i:i+batch_size].to(DEVICE)
            logits, emb = model(xb, return_embedding=True)
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_embeddings.append(emb.cpu().numpy())
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
    return (test_labels.numpy(), np.concatenate(all_preds),
            np.concatenate(all_probs), np.concatenate(all_embeddings))

DATASET_FILES = {
    "IndianPines": "preprocessed/IndianPines.npz",
    "PaviaUniversity": "preprocessed/PaviaUniversity.npz",
    "Salinas": "preprocessed/Salinas.npz",
    "KSC": "preprocessed/KSC.npz",
    "Botswana": "preprocessed/Botswana.npz",
    "Houston2013_Full15": "preprocessed/Houston2013_Full15.npz",
}

os.makedirs("artifacts", exist_ok=True)

for name, path in DATASET_FILES.items():
    print(f"\n=== Extracting artifacts: {name} ===")
    d = np.load(path)
    train_tokens = torch.tensor(d["train_tokens"], dtype=torch.float32)
    train_labels = torch.tensor(d["train_labels"] - 1, dtype=torch.long)
    val_tokens = torch.tensor(d["val_tokens"], dtype=torch.float32)
    val_labels = torch.tensor(d["val_labels"] - 1, dtype=torch.long)
    test_tokens = torch.tensor(d["test_tokens"], dtype=torch.float32)
    test_labels = torch.tensor(d["test_labels"] - 1, dtype=torch.long)
    k_dim = train_tokens.shape[-1]
    n_classes = int(train_labels.max().item()) + 1
    model = train_full_config(k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels)
    y_true, y_pred, y_proba, embeddings = extract_artifacts(model, test_tokens, test_labels)
    oa = accuracy_score(y_true, y_pred)
    print(f"  {name}: OA={oa:.4f} (retrained seed=42, for artifact extraction)")
    np.savez(f"artifacts/{name}_artifacts.npz",
             y_true=y_true, y_pred=y_pred, y_proba=y_proba, embeddings=embeddings)
    print(f"  saved artifacts/{name}_artifacts.npz")

print("\nAll six datasets' artifacts extracted and saved.")


=== Extracting artifacts: IndianPines ===
  IndianPines: OA=0.8854 (retrained seed=42, for artifact extraction)
  saved artifacts/IndianPines_artifacts.npz

=== Extracting artifacts: PaviaUniversity ===
  PaviaUniversity: OA=0.9853 (retrained seed=42, for artifact extraction)
  saved artifacts/PaviaUniversity_artifacts.npz

=== Extracting artifacts: Salinas ===
  Salinas: OA=0.9918 (retrained seed=42, for artifact extraction)
  saved artifacts/Salinas_artifacts.npz

=== Extracting artifacts: KSC ===
  KSC: OA=0.9741 (retrained seed=42, for artifact extraction)
  saved artifacts/KSC_artifacts.npz

=== Extracting artifacts: Botswana ===
  Botswana: OA=0.9512 (retrained seed=42, for artifact extraction)
  saved artifacts/Botswana_artifacts.npz

=== Extracting artifacts: Houston2013_Full15 ===
  Houston2013_Full15: OA=0.7420 (retrained seed=42, for artifact extraction)
  saved artifacts/Houston2013_Full15_artifacts.npz

All six datasets' artifacts extracted and saved.


In [14]:
# CELL 2 — Classification maps for all six datasets, using freshly-extracted
# predictions (consistent Full-config source across all six, unlike the
# confusion matrices which mixed Phase 2/3/5/6 origins)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

coord_data = np.load("preprocessed/test_coordinates.npz")

DATASET_DISPLAY_NAMES = {
    "IndianPines": "Indian Pines",
    "PaviaUniversity": "Pavia University",
    "Salinas": "Salinas",
    "KSC": "KSC",
    "Botswana": "Botswana",
    "Houston2013_Full15": "Houston 2013 (15-class)",
}

def plot_classification_map(y_pred, coords, scene_shape, n_classes, title, save_path):
    canvas = np.zeros(scene_shape, dtype=int) - 1
    for (r, c), label in zip(coords, y_pred):
        canvas[r, c] = label
    cmap = plt.cm.get_cmap("tab20", n_classes)
    colors = ["black"] + [cmap(i) for i in range(n_classes)]
    custom_cmap = mcolors.ListedColormap(colors)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(canvas + 1, cmap=custom_cmap, vmin=0, vmax=n_classes)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved {save_path}")

os.makedirs("figures", exist_ok=True)
for name, display_name in DATASET_DISPLAY_NAMES.items():
    artifacts = np.load(f"artifacts/{name}_artifacts.npz")
    y_pred = artifacts["y_pred"]
    coords = coord_data[f"{name}_coords"]
    scene_shape = tuple(coord_data[f"{name}_shape"])
    n_classes = int(y_pred.max()) + 1

    plot_classification_map(
        y_pred, coords, scene_shape, n_classes,
        title=f"Classification Map — {display_name} Full QNN-Transformer (seed 42)",
        save_path=f"figures/classmap_{name.lower()}_full.png",
    )

print("\nAll six classification maps saved.")

C:\Users\aipmu\AppData\Local\Temp\ipykernel_23956\1618943650.py:22: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get_cmap("tab20", n_classes)


  saved figures/classmap_indianpines_full.png
  saved figures/classmap_paviauniversity_full.png
  saved figures/classmap_salinas_full.png
  saved figures/classmap_ksc_full.png
  saved figures/classmap_botswana_full.png
  saved figures/classmap_houston2013_full15_full.png

All six classification maps saved.


In [ ]:
# CELL 3 — Extract classical-mirror embeddings (Indian Pines + Pavia only),
# to make the t-SNE comparison actually test the paper's quantum-vs-classical
# discriminability claim, per §10.2's original intent.

class ClassicalMirrorEncoder(nn.Module):
    """Same 4-dim bottleneck shape as the quantum encoder, no quantum circuit."""
    def __init__(self, k_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(k_dim, N_QUBITS), nn.Tanh(), nn.Linear(N_QUBITS, D_MODEL))
    def forward(self, tokens):
        return self.net(tokens)

class QuantFormerClassicalWithEmbeddings(nn.Module):
    """Identical to QuantFormerWithEmbeddings, but with the quantum encoder
    swapped for its classical mirror. Same transformer, same pooling, same head."""
    def __init__(self, k_dim, n_classes):
        super().__init__()
        self.encoder = ClassicalMirrorEncoder(k_dim)
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.transformer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True, dropout=0.0)
        self.classifier = nn.Linear(D_MODEL, n_classes)
    def forward(self, tokens, return_embedding=False):
        x = self.encoder(tokens)
        x = self.pos_enc(x)
        x = self.transformer(x)
        embedding = x.mean(dim=1)
        logits = self.classifier(embedding)
        if return_embedding:
            return logits, embedding
        return logits

def get_param_groups_classical(model):
    # no quantum weights to exclude here, but keep the same structure/signature
    # for consistency with the quantum training function
    return [{"params": model.parameters(), "weight_decay": WEIGHT_DECAY}]

def train_classical_config(k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels):
    torch.manual_seed(SEED)
    model = QuantFormerClassicalWithEmbeddings(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups_classical(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    best_val_acc, best_state = -1, None
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(val_labels, model(val_tokens.to(DEVICE)).argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

# ---- Run for the two fidelity checkpoints only ----
CLASSICAL_DATASET_FILES = {
    "IndianPines": "preprocessed/IndianPines.npz",
    "PaviaUniversity": "preprocessed/PaviaUniversity.npz",
}

for name, path in CLASSICAL_DATASET_FILES.items():
    print(f"\n=== Extracting CLASSICAL-MIRROR artifacts: {name} ===")
    d = np.load(path)
    train_tokens = torch.tensor(d["train_tokens"], dtype=torch.float32)
    train_labels = torch.tensor(d["train_labels"] - 1, dtype=torch.long)
    val_tokens = torch.tensor(d["val_tokens"], dtype=torch.float32)
    val_labels = torch.tensor(d["val_labels"] - 1, dtype=torch.long)
    test_tokens = torch.tensor(d["test_tokens"], dtype=torch.float32)
    test_labels = torch.tensor(d["test_labels"] - 1, dtype=torch.long)
    k_dim = train_tokens.shape[-1]
    n_classes = int(train_labels.max().item()) + 1

    model = train_classical_config(k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels)
    y_true, y_pred, y_proba, embeddings = extract_artifacts(model, test_tokens, test_labels)
    oa = accuracy_score(y_true, y_pred)
    print(f"  {name} [classical mirror]: OA={oa:.4f}")

    np.savez(f"artifacts/{name}_classical_artifacts.npz",
             y_true=y_true, y_pred=y_pred, y_proba=y_proba, embeddings=embeddings)
    print(f"  saved artifacts/{name}_classical_artifacts.npz")

print("\nClassical-mirror artifacts extracted for both fidelity checkpoints.")


=== Extracting CLASSICAL-MIRROR artifacts: IndianPines ===
  IndianPines [classical mirror]: OA=0.9039
  saved artifacts/IndianPines_classical_artifacts.npz

=== Extracting CLASSICAL-MIRROR artifacts: PaviaUniversity ===
  PaviaUniversity [classical mirror]: OA=0.9896
  saved artifacts/PaviaUniversity_classical_artifacts.npz

Classical-mirror artifacts extracted for both fidelity checkpoints.


In [ ]:
# CELL 4 (fixed) — Quantum vs. Classical embedding comparison: t-SNE + separability metrics
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split as sk_train_test_split
import matplotlib.pyplot as plt
import numpy as np

def fisher_discriminant_ratio(embeddings, labels):
    """Mean ratio of between-class variance to within-class variance, averaged
    across embedding dimensions — higher means more class-discriminative."""
    classes = np.unique(labels)
    overall_mean = embeddings.mean(axis=0)
    between_var = np.zeros(embeddings.shape[1])
    within_var = np.zeros(embeddings.shape[1])
    for c in classes:
        class_emb = embeddings[labels == c]
        class_mean = class_emb.mean(axis=0)
        n_c = len(class_emb)
        between_var += n_c * (class_mean - overall_mean) ** 2
        within_var += ((class_emb - class_mean) ** 2).sum(axis=0)
    between_var /= len(labels)
    within_var /= len(labels)
    return np.mean(between_var / (within_var + 1e-10))

def linear_probe_accuracy(embeddings, labels, seed=42):
    """Logistic regression on frozen embeddings, 80/20 split. max_iter increased
    to 5000 (from sklearn's default 100 / our original 1000) after a
    ConvergenceWarning on Indian Pines — ensures a fair, fully-converged
    comparison between quantum and classical embeddings, not an artifact of
    early stopping."""
    X_train, X_test, y_train, y_test = sk_train_test_split(
        embeddings, labels, test_size=0.2, random_state=seed, stratify=labels
    )
    clf = LogisticRegression(max_iter=5000)
    clf.fit(X_train, y_train)
    return clf.score(X_test, y_test)

def subsample_stratified(y_true, max_n=10000, seed=42):
    if len(y_true) <= max_n:
        return np.arange(len(y_true))
    rng = np.random.default_rng(seed)
    idx_per_class = []
    for c in np.unique(y_true):
        class_idx = np.where(y_true == c)[0]
        n_take = max(1, int(len(class_idx) * max_n / len(y_true)))
        idx_per_class.append(rng.choice(class_idx, size=min(n_take, len(class_idx)), replace=False))
    return np.concatenate(idx_per_class)

FIDELITY_CHECKPOINTS = ["IndianPines", "PaviaUniversity"]
DISPLAY_NAMES = {"IndianPines": "Indian Pines", "PaviaUniversity": "Pavia University"}
separability_results = {}

for name in FIDELITY_CHECKPOINTS:
    display = DISPLAY_NAMES[name]
    print(f"\n=== {display}: Quantum vs. Classical embedding comparison ===")

    q_data = np.load(f"artifacts/{name}_artifacts.npz")
    c_data = np.load(f"artifacts/{name}_classical_artifacts.npz")

    q_emb, q_labels = q_data["embeddings"], q_data["y_true"]
    c_emb, c_labels = c_data["embeddings"], c_data["y_true"]

    q_silhouette = silhouette_score(q_emb, q_labels, sample_size=min(10000, len(q_labels)), random_state=42)
    c_silhouette = silhouette_score(c_emb, c_labels, sample_size=min(10000, len(c_labels)), random_state=42)
    q_fisher = fisher_discriminant_ratio(q_emb, q_labels)
    c_fisher = fisher_discriminant_ratio(c_emb, c_labels)
    q_probe = linear_probe_accuracy(q_emb, q_labels)
    c_probe = linear_probe_accuracy(c_emb, c_labels)

    separability_results[name] = {
        "quantum": {"silhouette": float(q_silhouette), "fisher_ratio": float(q_fisher), "linear_probe_acc": float(q_probe)},
        "classical": {"silhouette": float(c_silhouette), "fisher_ratio": float(c_fisher), "linear_probe_acc": float(c_probe)},
    }

    print(f"  Silhouette score    — Quantum: {q_silhouette:.4f}  Classical: {c_silhouette:.4f}")
    print(f"  Fisher discr. ratio — Quantum: {q_fisher:.4f}  Classical: {c_fisher:.4f}")
    print(f"  Linear probe acc.   — Quantum: {q_probe:.4f}  Classical: {c_probe:.4f}")

    idx = subsample_stratified(q_labels, max_n=10000)
    print(f"  running t-SNE on {len(idx)} points each...")
    q_proj = TSNE(n_components=2, random_state=42, init="pca").fit_transform(q_emb[idx])
    c_proj = TSNE(n_components=2, random_state=42, init="pca").fit_transform(c_emb[idx])

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    axes[0].scatter(q_proj[:, 0], q_proj[:, 1], c=q_labels[idx], cmap="tab20", s=5, alpha=0.7)
    axes[0].set_title(f"Quantum Encoder — {display}\n(silhouette={q_silhouette:.3f}, probe acc={q_probe:.3f})")
    axes[0].set_xlabel("t-SNE Dim 1"); axes[0].set_ylabel("t-SNE Dim 2")

    axes[1].scatter(c_proj[:, 0], c_proj[:, 1], c=c_labels[idx], cmap="tab20", s=5, alpha=0.7)
    axes[1].set_title(f"Classical Mirror — {display}\n(silhouette={c_silhouette:.3f}, probe acc={c_probe:.3f})")
    axes[1].set_xlabel("t-SNE Dim 1"); axes[1].set_ylabel("t-SNE Dim 2")

    fig.suptitle(f"Learned Embedding Comparison — {display} (seed 42)", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"figures/tsne_comparison_{name.lower()}.png", dpi=150)
    plt.close(fig)
    print(f"  saved figures/tsne_comparison_{name.lower()}.png")

with open("phase7_separability_results.json", "w") as f:
    json.dump(separability_results, f, indent=2)
print("\nSaved phase7_separability_results.json (overwrites previous run with converged probe accuracy)")


=== Indian Pines: Quantum vs. Classical embedding comparison ===
  Silhouette score    — Quantum: 0.1733  Classical: 0.1987
  Fisher discr. ratio — Quantum: 3.1751  Classical: 3.1753
  Linear probe acc.   — Quantum: 0.9335  Classical: 0.9524
  running t-SNE on 8200 points each...
  saved figures/tsne_comparison_indianpines.png

=== Pavia University: Quantum vs. Classical embedding comparison ===
  Silhouette score    — Quantum: 0.4340  Classical: 0.3911
  Fisher discr. ratio — Quantum: 4.9725  Classical: 4.3605
  Linear probe acc.   — Quantum: 0.9918  Classical: 0.9969
  running t-SNE on 9995 points each...
  saved figures/tsne_comparison_paviauniversity.png

Saved phase7_separability_results.json (overwrites previous run with converged probe accuracy)


In [ ]:
# CELL 5 — Precision/Recall/F1, Balanced Accuracy, Producer's/User's Accuracy,
# parameter counts, and per-sample inference time — all six datasets, Full config

from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score
import time

ALL_DATASETS = ["IndianPines", "PaviaUniversity", "Salinas", "KSC", "Botswana", "Houston2013_Full15"]

# --- Step 1: gather y_true/y_pred for all six, from whichever source has them ---
def get_predictions(name):
    if name in ["IndianPines", "PaviaUniversity"]:
        a = np.load(f"artifacts/{name}_artifacts.npz")
        return a["y_true"], a["y_pred"]
    elif name in ["Salinas", "KSC", "Botswana"]:
        with open("phase5_ablation_predictions.json") as f:
            preds = json.load(f)
        d = np.load(f"preprocessed/{name}.npz")
        y_true = d["test_labels"] - 1
        y_pred = np.array(preds[name]["Full"]["42"])
        return y_true, y_pred
    elif name == "Houston2013_Full15":
        with open("phase6_houston15_predictions.json") as f:
            preds = json.load(f)
        d = np.load(f"preprocessed/{name}.npz")
        y_true = d["test_labels"] - 1
        y_pred = np.array(preds["Full"]["unaugmented"]["42"])
        return y_true, y_pred

full_metrics = {}
for name in ALL_DATASETS:
    y_true, y_pred = get_predictions(name)
    n_classes = int(max(y_true.max(), y_pred.max())) + 1

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    # Per-class precision/recall = User's Accuracy / Producer's Accuracy (remote-sensing terminology)
    per_class_p, per_class_r, per_class_f1, per_class_support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=list(range(n_classes)), zero_division=0
    )

    full_metrics[name] = {
        "macro_precision": float(macro_p), "macro_recall": float(macro_r), "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p), "weighted_recall": float(weighted_r), "weighted_f1": float(weighted_f1),
        "balanced_accuracy": float(bal_acc),
        "per_class_users_accuracy": per_class_p.tolist(),      # = precision
        "per_class_producers_accuracy": per_class_r.tolist(),  # = recall
        "per_class_f1": per_class_f1.tolist(),
        "per_class_support": per_class_support.tolist(),
    }
    print(f"{name}: macro-F1={macro_f1:.4f}  weighted-F1={weighted_f1:.4f}  balanced-acc={bal_acc:.4f}")

# --- Step 2: parameter counts per config (no training needed, just instantiate) ---
print(f"\n{'='*60}\nParameter counts by config\n{'='*60}")
param_counts = {}
sample_k = {name: get_predictions(name) for name in ALL_DATASETS}  # reuse to get n_classes below

for name in ALL_DATASETS:
    d = np.load(f"preprocessed/{name}.npz")
    k_dim = d["train_tokens"].shape[-1]
    n_classes = int(d["train_labels"].max())

    full_model = QuantFormerWithEmbeddings(k_dim, n_classes)
    classical_model = QuantFormerClassicalWithEmbeddings(k_dim, n_classes)

    full_params = sum(p.numel() for p in full_model.parameters())
    classical_params = sum(p.numel() for p in classical_model.parameters())

    param_counts[name] = {"full_config": full_params, "classical_mirror": classical_params}
    print(f"{name}: Full={full_params}, w/o QNN={classical_params}")

# --- Step 3: per-sample inference time (Full config, seed 42, on this hardware) ---
print(f"\n{'='*60}\nPer-sample inference time (Full config)\n{'='*60}")
inference_times = {}
for name in ALL_DATASETS:
    d = np.load(f"preprocessed/{name}.npz")
    test_tokens = torch.tensor(d["test_tokens"], dtype=torch.float32)
    k_dim = test_tokens.shape[-1]
    n_classes = int(d["train_labels"].max())

    torch.manual_seed(42)
    model = QuantFormerWithEmbeddings(k_dim, n_classes).to(DEVICE).eval()

    batch = test_tokens[:256].to(DEVICE)  # warmup
    with torch.no_grad():
        _ = model(batch)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    n_timed = min(1000, len(test_tokens))
    timed_batch = test_tokens[:n_timed].to(DEVICE)
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(timed_batch)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    per_sample_ms = (elapsed / n_timed) * 1000
    inference_times[name] = per_sample_ms
    print(f"{name}: {per_sample_ms:.4f} ms/sample (n={n_timed}, untrained weights — architecture-level timing only)")

# --- Save everything ---
with open("phase7_full_metrics.json", "w") as f:
    json.dump({"classification_metrics": full_metrics, "parameter_counts": param_counts,
               "inference_time_ms_per_sample": inference_times}, f, indent=2)
print("\nSaved phase7_full_metrics.json")

IndianPines: macro-F1=0.8011  weighted-F1=0.8843  balanced-acc=0.7835
PaviaUniversity: macro-F1=0.9799  weighted-F1=0.9854  balanced-acc=0.9836
Salinas: macro-F1=0.9957  weighted-F1=0.9918  balanced-acc=0.9955
KSC: macro-F1=0.9593  weighted-F1=0.9741  balanced-acc=0.9630
Botswana: macro-F1=0.9470  weighted-F1=0.9507  balanced-acc=0.9418
Houston2013_Full15: macro-F1=0.7393  weighted-F1=0.7460  balanced-acc=0.7566

Parameter counts by config
IndianPines: Full=34984, w/o QNN=34960
PaviaUniversity: Full=34445, w/o QNN=34421
Salinas: Full=34900, w/o QNN=34876
KSC: Full=34705, w/o QNN=34681
Botswana: Full=34770, w/o QNN=34746
Houston2013_Full15: Full=34835, w/o QNN=34811

Per-sample inference time (Full config)
IndianPines: 0.2983 ms/sample (n=1000, untrained weights — architecture-level timing only)
PaviaUniversity: 0.2787 ms/sample (n=1000, untrained weights — architecture-level timing only)
Salinas: 0.2775 ms/sample (n=1000, untrained weights — architecture-level timing only)
KSC: 0.2803 